# Zaks et al. 2012 — strict-source MxlPy port and figure reruns

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mxlmodels import get_zaks2012
from mxlpy import Model, Simulator, make_protocol

In [ ]:
from mxlpy.integrators import Scipy
from functools import partial


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
import numpy as np

def cat_pam(vin, nreps):
    return np.tile(np.asarray(vin, dtype=float), nreps)

def setup_pam_intensities(act, Isat=7000, flashlength=0.5, simtype=1):
    Imeasure = 0.001

    PAMintens = np.array([act, Isat, act], dtype=float)
    PAMdurat28 = np.array([28, flashlength, 2 - flashlength], dtype=float)
    PAMdurat58 = np.array([58, flashlength, 2 - flashlength], dtype=float)
    PAMdurat15 = np.array([13.0, flashlength, 2 - flashlength], dtype=float)
    PAMdurat7 = np.array([7.0, flashlength, 2 - flashlength], dtype=float)

    offintens = np.array([Imeasure, Isat, Imeasure], dtype=float)
    offintensnosat = np.array([Imeasure, Isat, Imeasure], dtype=float)
    fidx = np.array([0, 1, 0], dtype=int)

    if simtype == 1:
        LightIntensities = np.concatenate([
            offintens,
            [Imeasure, Isat],
            cat_pam(PAMintens, 23),
            cat_pam(offintensnosat, 15),
        ])

        durat = np.concatenate([
            PAMdurat58,
            [60, flashlength],
            cat_pam(PAMdurat7, 2),
            cat_pam(PAMdurat15, 10),
            cat_pam(PAMdurat28, 4),
            cat_pam(PAMdurat58, 7),
            cat_pam(PAMdurat7, 2),
            cat_pam(PAMdurat28, 3),
            cat_pam(PAMdurat58, 10),
        ])

        flashyesno = np.concatenate([
            fidx,
            [0, 1],
            cat_pam(fidx, 30),
        ])

    elif simtype == 2:
        raise ValueError("this case doesnt work right now")
    else:
        raise ValueError("simtype must be 1 or 2")

    flashidx = np.where(flashyesno != 0)[0]
    return LightIntensities, durat, flashidx

In [ ]:
def calc_pam_vals2(
    fluo_result: pd.Series, protocol: pd.DataFrame, pfd_str: str = "pfd", sat_pulse: float = 5000
) -> pd.DataFrame:
    """Calculate PAM values from fluorescence data.

    Use the fluorescence data from a PAM protocol to calculate Fm, NPQ. To find the Fm values, the protocol used for simulation is seperated into ranges between each saturating pulse. Then the maximum fluorescence value within each range is taken as Fm. Thes are then used to calculate NPQ.

    Args:
        fluo_result (pd.Series): Fluorescence data as a pd.Series from mxlpy simulation.
        protocol (pd.DataFrame): PAM protocol used for simulation. Created using make_protocol from mxlpy.
        pfd_str (str): The name of the PPFD parameter in the protocol.
        sat_pulse (float, optional): The threshold for saturating pulse in the protocol. Defaults to 2000.

    Returns:
         pd.Series: NPQ as pd.Series
    """    
    fluo_result = fluo_result.div(fluo_result.max()).fillna(0)

    # Step 0: Ensure that protocol has same index as fluo_results
    name = fluo_result.name
    # common_index = protocol.index.union(fluo_result.index).sort_values()
    merged = pd.concat([protocol, fluo_result], axis = 1).sort_index().bfill()
    # Step 1: Find all periods that has pfd = SP

    grp = (merged[pfd_str] != merged[pfd_str].shift()).cumsum()
    groups_sp = merged[grp.isin(grp[merged[pfd_str] == sat_pulse])]
    peaks = [g for _, g in groups_sp.groupby(grp[groups_sp.index])]

    # Step 2: Fm is the max fluo value of each SP period

    Fm = pd.Series(
        [k[name].max() for k in peaks],
        index= [k[name].idxmax() for k in peaks],
    )
    Fm.name = "Fm"
    
    # Calculate NPQ
    NPQ = (Fm.iloc[0] - Fm) / Fm if len(Fm) > 0 else pd.Series(dtype=float)
    NPQ.name = "Non-Photochemical Quenching (NPQ)"
    
    return NPQ

In [ ]:
LightIntensities, durat, flashidx = setup_pam_intensities(act=1000)

protocol = make_protocol(
    [
        (i, {"LightIntensity": k}) for i,k in zip(durat, LightIntensities)
    ] 
)

res_1000_WT = Simulator(get_zaks2012(),
                integrator=partial(Scipy, method="Radau",   # or "BDF" for stiff problems
                rtol=1e-8,
                atol=1e-10,
            ),
                ).simulate_protocol(protocol, time_points_per_step=100).get_result().unwrap_or_err().get_combined()



res_1000_npq4 = Simulator(get_zaks2012().update_parameters({"qtrigg1":0, "qtrigg3":1}),
                integrator=partial(Scipy, method="Radau",   # or "BDF" for stiff problems
                rtol=1e-8,
                atol=1e-10,
            ),
                ).simulate_protocol(protocol, time_points_per_step=100).get_result().unwrap_or_err().get_combined()

protocol.index = protocol.index.total_seconds()

fluo_1000_WT = res_1000_WT["fluorescenceyield"]
PAM_1000_WT = calc_pam_vals2( fluo_1000_WT,  protocol, "LightIntensity", 7000)
fluo_1000_npq4 = res_1000_npq4["fluorescenceyield"]
PAM_1000_npq4 = calc_pam_vals2( fluo_1000_npq4,  protocol, "LightIntensity", 7000)


In [ ]:
LightIntensities, durat, flashidx = setup_pam_intensities(act=100)

protocol = make_protocol(
    [
        (i, {"LightIntensity": k}) for i,k in zip(durat, LightIntensities)
    ] 
)

res_100_WT = Simulator(get_zaks2012(),
                integrator=partial(Scipy, method="Radau",   # or "BDF" for stiff problems
                rtol=1e-8,
                atol=1e-10,
            ),
                ).simulate_protocol(protocol, time_points_per_step=100).get_result().unwrap_or_err().get_combined()


res_100_npq4 = Simulator(get_zaks2012().update_parameters({"qtrigg1":0, "qtrigg3":0}),
                integrator=partial(Scipy, method="Radau",   # or "BDF" for stiff problems
                rtol=1e-8,
                atol=1e-10,
            ),
                ).simulate_protocol(protocol, time_points_per_step=100).get_result().unwrap_or_err().get_combined()

protocol.index = protocol.index.total_seconds()

fluo_100_WT = res_100_WT["fluorescenceyield"]
PAM_100_WT = calc_pam_vals2( fluo_100_WT,  protocol, "LightIntensity", 7000)
fluo_100_npq4 = res_100_npq4["fluorescenceyield"]
PAM_100_npq4 = calc_pam_vals2( fluo_100_npq4,  protocol, "LightIntensity", 7000)

In [ ]:
ncol = 3
nrow = 1
fig, axs = plt.subplots(ncols=ncol, nrows=nrow, figsize = (ncol*5, nrow*5), sharex = True)

axs[0].plot(fluo_1000_npq4.index, fluo_1000_npq4.values, color = 'red', label = 'npq4')
axs[0].plot(fluo_1000_WT.index, fluo_1000_WT.values, color = 'black', label = 'WT')
axs[0].set_title(r"100 $\mu$mol photon m$^{-2}$ s$^{-1}$")
axs[0].set_ylabel("Fluo")

axs[1].plot(PAM_1000_npq4.index, PAM_1000_npq4.values, color = 'red', label = 'npq4')
axs[1].plot(PAM_1000_WT.index, PAM_1000_WT.values, color = 'black', label = 'WT')
axs[1].set_title(r"1000 $\mu$mol photon m$^{-2}$ s$^{-1}$")
axs[1].set_ylabel("NPQ")

axs[2].plot(PAM_1000_npq4.index, PAM_1000_WT.values - PAM_1000_npq4.values, color = 'black', label = 'qE')
axs[2].set_title(r"1000 $\mu$mol photon m$^{-2}$ s$^{-1}$")
axs[2].set_ylabel("NPQ")

for ax in axs.flatten():
    ax.legend()
    ax.set_xlabel('Time(s)')

fig.text(0.5, 1, "Figure 3", fontsize = 15)
plt.show()

In [ ]:
ncol = 2
nrow = 1
fig, axs = plt.subplots(ncols=ncol, nrows=nrow, figsize = (ncol*5, nrow*5), sharex = True)

axs[0].plot(PAM_100_WT.index, PAM_100_WT.values  - PAM_100_npq4.values, color = 'black', label = 'Simulation', linewidth = 4)
axs[0].set_title(r"100 $\mu$mol photon m$^{-2}$ s$^{-1}$")
axs[0].set_ylim(-0.1,1.5)
axs[1].plot(PAM_1000_WT.index, PAM_1000_WT.values - PAM_1000_npq4.values, color = 'black', label = 'Simulation', linewidth = 4)
axs[1].set_title(r"1000 $\mu$mol photon m$^{-2}$ s$^{-1}$")
axs[1].set_ylim(-0.1,2)

for ax in axs.flatten():
    ax.legend()
    ax.set_xlabel('Time(s)')

fig.text(0.47, 1, "Figure 4", fontsize = 15)

plt.show()

In [ ]:
ncol = 2
nrow = 1
fig, axs = plt.subplots(ncols=ncol, nrows=nrow, figsize = (ncol*5, nrow*5), sharex = True, sharey=True)

axs[0].plot(res_100_WT.index, res_100_WT["pH_lumen"], color = 'black', label = 'WT')
axs[0].plot(res_100_npq4.index, res_100_npq4["pH_lumen"], color = 'red', label = 'npq4')
axs[0].set_title(r"100 $\mu$mol photon m$^{-2}$ s$^{-1}$")
axs[0].set_ylabel("pH lumen")
axs[1].plot(res_1000_WT.index, res_1000_WT["pH_lumen"], color = 'black', label = 'WT')
axs[1].plot(res_1000_npq4.index, res_1000_npq4["pH_lumen"], color = 'red', label = 'npq4')
axs[1].set_title(r"1000 $\mu$mol photon m$^{-2}$ s$^{-1}$")

for ax in axs.flatten():
    ax.legend()
    ax.set_xlabel('Time(s)')

fig.text(0.45, 1, "Figure 5", fontsize = 15)
plt.show()

In [ ]:
LightIntensities, durat, flashidx = setup_pam_intensities(act=1000)

protocol = make_protocol(
    [
        (i, {"LightIntensity": k}) for i,k in zip(durat, LightIntensities)
    ] 
)

res_kQ_3e9 = Simulator(get_zaks2012(),
                integrator=partial(Scipy, method="Radau",   # or "BDF" for stiff problems
                rtol=1e-8,
                atol=1e-10,
            ),
                ).simulate_protocol(protocol, time_points_per_step=100).get_result().unwrap_or_err().get_combined()

res_kQ_1e10 = Simulator(get_zaks2012().update_parameter("kQ", 1e10),
                integrator=partial(Scipy, method="Radau",   # or "BDF" for stiff problems
                rtol=1e-8,
                atol=1e-10,
            ),
                ).simulate_protocol(protocol, time_points_per_step=100).get_result().unwrap_or_err().get_combined()
res_kQ_3e10 = Simulator(get_zaks2012().update_parameter("kQ", 3e10),
                integrator=partial(Scipy, method="Radau",   # or "BDF" for stiff problems
                rtol=1e-8,
                atol=1e-10,
            ),
                ).simulate_protocol(protocol, time_points_per_step=100).get_result().unwrap_or_err().get_combined()
res_kQ_1e11 = Simulator(get_zaks2012().update_parameter("kQ", 1e11),
                integrator=partial(Scipy, method="Radau",   # or "BDF" for stiff problems
                rtol=1e-8,
                atol=1e-10,
            ),
                ).simulate_protocol(protocol, time_points_per_step=100).get_result().unwrap_or_err().get_combined()


protocol.index = protocol.index.total_seconds()

fluo_kQ_3e9 = res_kQ_3e9["fluorescenceyield"]
PAM_kQ_3e9 = calc_pam_vals2( fluo_kQ_3e9,  protocol, "LightIntensity", 7000)

fluo_kQ_1e10 = res_kQ_1e10["fluorescenceyield"]
PAM_kQ_1e10 = calc_pam_vals2( fluo_kQ_1e10,  protocol, "LightIntensity", 7000)

fluo_kQ_3e10 = res_kQ_3e10["fluorescenceyield"]
PAM_kQ_3e10 = calc_pam_vals2( fluo_kQ_3e10,  protocol, "LightIntensity", 7000)

fluo_kQ_1e11 = res_kQ_1e11["fluorescenceyield"]
PAM_kQ_1e11 = calc_pam_vals2( fluo_kQ_1e11,  protocol, "LightIntensity", 7000)


In [ ]:
fig, ax = plt.subplots()

res_list= [res_kQ_3e9, res_kQ_1e10, res_kQ_3e10, res_kQ_1e11]
label_list = ["kQ = 3e9", "kQ = 1e10", "kQ = 3e10", "kQ = 1e11"]
colors = ["#960000", "#008100", "#ff4c00", "#0030d1"]

for res, label, color in zip(res_list, label_list, colors):
    ax.plot(res.index, res["pH_lumen"], label = label, color = color, linewidth = 4)

ax.legend()
ax.set_xlabel("Time (s)")
ax.set_ylabel("pH lumen")

fig.text(0.45, 1, "Figure 6", fontsize = 15)